# 21 - Build Distributable Packed Recovered L78

Notebook para montar uma pasta distribuível do modelo, no formato `baixei e executei`.

Este notebook não treina e não altera pesos. Ele copia ou referencia os artefatos necessários e cria um runtime CLI com verificação de integridade.

Resultado esperado:

```text
build/distributable_packed_recovered_l78/
  student_pruned/
  adapter_recovered_l78/
  packed/
  runtime/
  README.md
  package_manifest.json
```

Nota técnica: este pacote é executável, mas ainda é `materialized_hybrid`, não runtime ternário nativo. O próximo marco para leveza real é remover MLP denso duplicado ou executar packed direto.

In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, shutil, subprocess

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
RUNTIME_TEMPLATE_DIR = PROJECT_ROOT / 'runtime' / 'packed_recovered_l78'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'packed_recovery'
DIST_DIR = PROJECT_ROOT / 'build' / 'distributable_packed_recovered_l78'

STUDENT_DIR = CHECKPOINTS_DIR / 'student_pruned'
ADAPTER_DIR = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_t4_resume'
PACKAGE_CANDIDATES = [EXPORTS_DIR / 'ternary_mlp_l78', DEPLOY_DIR]
PACKED_SOURCE_DIR = next((p for p in PACKAGE_CANDIDATES if (p / 'manifest.json').exists() and (p / 'mlp_ternary_2bit.bin').exists()), None)

EXPECTED_PACKED_SHA256 = '0b6849fe3ceaab615a4e54f63f5a96e9d9d431ba03b7b791a7ab0638ea174116'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DIST_DIR =', DIST_DIR)
print('STUDENT_DIR =', STUDENT_DIR)
print('ADAPTER_DIR =', ADAPTER_DIR)
print('PACKED_SOURCE_DIR =', PACKED_SOURCE_DIR)
print('RUNTIME_TEMPLATE_DIR =', RUNTIME_TEMPLATE_DIR)

In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def dir_size_bytes(path):
    path = Path(path)
    if not path.exists():
        return None
    if path.is_file():
        return path.stat().st_size
    return sum(item.stat().st_size for item in path.rglob('*') if item.is_file())

def format_bytes(n):
    if n is None:
        return None
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024

def copytree_clean(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    shutil.copytree(src, dst)

def copy_file_clean(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)

required_paths = [STUDENT_DIR, ADAPTER_DIR, RUNTIME_TEMPLATE_DIR]
if PACKED_SOURCE_DIR is None:
    raise FileNotFoundError('Packed source not found in exports/ternary_mlp_l78 or deploy')
required_paths.extend([PACKED_SOURCE_DIR / 'manifest.json', PACKED_SOURCE_DIR / 'mlp_ternary_2bit.bin'])
missing = [str(p) for p in required_paths if not Path(p).exists()]
if missing:
    raise FileNotFoundError('Missing required artifacts:\n' + '\n'.join(missing))

packed_sha = sha256_file(PACKED_SOURCE_DIR / 'mlp_ternary_2bit.bin')
assert packed_sha == EXPECTED_PACKED_SHA256, packed_sha
print('packed_sha ok:', packed_sha)

In [ ]:
# Monta a pasta distribuível. COPY_MODE pode ser 'copy' ou 'symlink'.
COPY_MODE = os.environ.get('RPT_DISTRIBUTABLE_COPY_MODE', 'copy')
if COPY_MODE not in {'copy', 'symlink'}:
    raise ValueError('COPY_MODE must be copy or symlink')

if DIST_DIR.exists():
    shutil.rmtree(DIST_DIR)
DIST_DIR.mkdir(parents=True, exist_ok=True)

def place_dir(src, dst):
    if COPY_MODE == 'symlink':
        dst.symlink_to(Path(src).resolve(), target_is_directory=True)
    else:
        copytree_clean(src, dst)

place_dir(STUDENT_DIR, DIST_DIR / 'student_pruned')
place_dir(ADAPTER_DIR, DIST_DIR / 'adapter_recovered_l78')
copytree_clean(RUNTIME_TEMPLATE_DIR, DIST_DIR / 'runtime')
(DIST_DIR / 'packed').mkdir(parents=True, exist_ok=True)
copy_file_clean(PACKED_SOURCE_DIR / 'manifest.json', DIST_DIR / 'packed' / 'manifest.json')
copy_file_clean(PACKED_SOURCE_DIR / 'mlp_ternary_2bit.bin', DIST_DIR / 'packed' / 'mlp_ternary_2bit.bin')

sizes = {
    'student_pruned': dir_size_bytes(DIST_DIR / 'student_pruned'),
    'adapter_recovered_l78': dir_size_bytes(DIST_DIR / 'adapter_recovered_l78'),
    'packed': dir_size_bytes(DIST_DIR / 'packed'),
    'runtime': dir_size_bytes(DIST_DIR / 'runtime'),
    'total': dir_size_bytes(DIST_DIR),
}
print(json.dumps({k: {'bytes': v, 'human': format_bytes(v)} for k, v in sizes.items()}, ensure_ascii=False, indent=2))

In [ ]:
manifest = {
    'name': 'rpt-packed-recovered-l78',
    'version': '0.1.0',
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'copy_mode': COPY_MODE,
    'runtime_kind': 'materialized_hybrid',
    'is_native_ternary_runtime': False,
    'load_order': 'student_pruned -> packed MLP materialization -> recovered adapter',
    'expected_packed_sha256': EXPECTED_PACKED_SHA256,
    'actual_packed_sha256': sha256_file(DIST_DIR / 'packed' / 'mlp_ternary_2bit.bin'),
    'expected_packed_mlp_tensors': 78,
    'validated_local_accuracy': {'benchmark': 'gemma4_100_questions', 'correct': 100, 'total': 100, 'accuracy': 1.0},
    'source_paths': {
        'student_pruned': str(STUDENT_DIR),
        'adapter_recovered_l78': str(ADAPTER_DIR),
        'packed_source': str(PACKED_SOURCE_DIR),
    },
    'distribution_layout': {
        'student_pruned': 'student_pruned/',
        'adapter_recovered_l78': 'adapter_recovered_l78/',
        'packed_manifest': 'packed/manifest.json',
        'packed_bin': 'packed/mlp_ternary_2bit.bin',
        'runtime_cli': 'runtime/run_inference.py',
    },
    'sizes': {k: {'bytes': v, 'human': format_bytes(v)} for k, v in sizes.items()},
    'known_limitations': [
        'Still requires dense student_pruned base.',
        'Packed MLP tensors are materialized into the base before inference.',
        'Not yet a native ternary kernel/runtime.',
    ],
}
(DIST_DIR / 'package_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

readme = f'''# RPT Packed Recovered L78

Executável local do modelo packed recovered L78.

## Instalação

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r runtime/requirements.txt
```

## Rodar

```bash
python runtime/run_inference.py --model-dir . --prompt "Qual a capital da França?"
```

## Estado

- Runtime: `materialized_hybrid`
- Runtime ternário nativo: `false`
- Acurácia local reproduzida: `100/100`
- Packed SHA: `{manifest['actual_packed_sha256']}`
- Tamanho total aproximado: `{manifest['sizes']['total']['human']}`

Este pacote é executável, mas ainda depende da base densa `student_pruned`. A próxima etapa de leveza real é remover duplicação dos MLPs densos ou implementar execução packed nativa.
'''
(DIST_DIR / 'README.md').write_text(readme, encoding='utf-8')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
result_path = RESULTS_DIR / f"21_build_distributable_packed_recovered_l78_{time.strftime('%Y%m%d_%H%M%S')}.json"
result_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('DIST_DIR:', DIST_DIR)
print('manifest:', DIST_DIR / 'package_manifest.json')
print('result:', result_path)

In [ ]:
# Smoke test opcional. Executa uma pergunta curta usando o runtime distribuível.
RUN_SMOKE_TEST = os.environ.get('RPT_RUN_DISTRIBUTABLE_SMOKE_TEST', '0') == '1'
if RUN_SMOKE_TEST:
    cmd = [
        sys.executable,
        str(DIST_DIR / 'runtime' / 'run_inference.py'),
        '--model-dir', str(DIST_DIR),
        '--prompt', 'Qual a capital da França?',
        '--max-new-tokens', '24',
        '--json',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Smoke test pulado. Para executar, defina RPT_RUN_DISTRIBUTABLE_SMOKE_TEST=1 antes desta célula.')

## Próximo marco

Depois que este pacote estiver executável para terceiros, a próxima etapa técnica deve ser uma destas:

1. `22_strip_dense_mlp_from_base_checkpoint`: criar checkpoint base sem os MLPs densos duplicados e materializar apenas do packed.
2. `22_native_ternary_mlp_runtime`: substituir `GemmaMLP` por módulo que executa o packed/ternário diretamente, sem reconstruir tensor float completo.

A opção 1 reduz distribuição e elimina redundância em disco. A opção 2 é a que pode reduzir memória e acelerar inferência de verdade.